# Chronos2 Multivariate Fine-Tuning

Train and evaluate Chronos2 foundation model with proper multivariate structure.

## 1. Setup and Configuration

In [1]:
import os
import sys
import json
import glob
import pandas as pd
import numpy as np
import torch
import random
from chronos import BaseChronosPipeline, Chronos2Pipeline
from tqdm.auto import tqdm
import logging
import shutil

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

# Setup paths
if os.path.basename(os.getcwd()) == 'notebooks':
    project_root = os.path.abspath('..')
else:
    project_root = os.getcwd()

if project_root not in sys.path:
    sys.path.append(project_root)

# Import sMAPE from your datamodule
from src.datamodule import smoothed_smape

BASE_DIR = project_root
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")
TEST_DIR = os.path.join(DATA_DIR, "test")
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [2]:
# Configuration
TARGET_COLS = ["high", "low", "close", "volume"]
OUTPUT_CHUNK_LENGTH = 10  # Prediction horizon
SEED = 827
MAX_NUMBER_FILES = 10

# Define covariates
PAST_COVARIATES = [
    'is_trading',
    'close_lag_adj_1', 'close_delta_adj_1', 'volume_adj_1',
    'close_lag_adj_2', 'close_delta_adj_2', 'volume_adj_2',
    'close_lag_adj_3', 'close_delta_adj_3', 'volume_adj_3',
    'close_lag_adj_4', 'close_delta_adj_4', 'volume_adj_4',
    'close_lag_adj_5', 'close_delta_adj_5', 'volume_adj_5',
    'close_lag_adj_6', 'close_delta_adj_6', 'volume_adj_6',
    'nearest_liquid_contract_close', 'cross_contract_mean',
    # Add future covariates here as they must be a subset of past covariates
    'time_to_delivery',
    'hour_of_day',
    'day_of_week',
    'week_of_year',
    'month',
    'is_weekend',
]

FUTURE_COVARIATES = [
    'time_to_delivery',
    'hour_of_day',
    'day_of_week',
    'week_of_year',
    'month',
    'is_weekend',
]

# ============================================================================
# Chronos2 Hyperparameters (adapted from LSTM best params)
# ============================================================================
# Load hyperparameters from your LSTM tuning results
best_params_path = os.path.join(RESULTS_DIR, "best_params_lstm.json")
with open(best_params_path, 'r') as f:
    lstm_params = json.load(f)

# Training configuration
INPUT_CHUNK_LENGTH = 24
BATCH_SIZE = lstm_params['batch_size']
BASE_LR = lstm_params['lr'] / 50  # (conservative for fine-tuning)
MIN_LR = BASE_LR / 100    # Floor for LR decay

# Warmup phase (progressive unfreezing)
WARMUP_EPOCHS_PHASE1 = 10   # Freeze all but head
WARMUP_EPOCHS_PHASE2 = 10   # Unfreeze 2 additional layers
WARMUP_LR_PHASE1 = BASE_LR
WARMUP_LR_PHASE2 = BASE_LR / 2

# Burst training configuration
BURST_STEPS = 200          # Steps per training burst
MAX_BURSTS = 20          # Maximum number of bursts
PATIENCE = 5               # Early stopping patience
LR_DECAY_FACTOR = 0.75      # Multiply LR by this on plateau

# Regularization
WEIGHT_DECAY = 1e-4        # L2 regularization
GRADIENT_CLIP = 1.0        # Max gradient norm
GRAD_ACCUMULATION = 4      # Gradient accumulation steps (i.e effective batch = 64*4 = 256)

logging.info(f"Loaded hyperparameters from LSTM tuning:")
logging.info(f"  BATCH_SIZE: {BATCH_SIZE} (from LSTM)")
logging.info(f"  BASE_LR: {BASE_LR} (LSTM lr / 50)")
logging.info(f"  CONTEXT_LENGTH: {INPUT_CHUNK_LENGTH}")
logging.info(f"  GRAD_ACCUMULATION: {GRAD_ACCUMULATION} (effective batch: {BATCH_SIZE * GRAD_ACCUMULATION})")
logging.info(f"")
logging.info(f"Warmup strategy:")
logging.info(f"  Phase 1: {WARMUP_EPOCHS_PHASE1} epochs, freeze all but head, LR={WARMUP_LR_PHASE1}")
logging.info(f"  Phase 2: {WARMUP_EPOCHS_PHASE2} epochs, unfreeze 2 layers, LR={WARMUP_LR_PHASE2}")
logging.info(f"")
logging.info(f"Target columns: {TARGET_COLS}")
logging.info(f"Past covariates: {len(PAST_COVARIATES)} features")
logging.info(f"Future covariates: {len(FUTURE_COVARIATES)} features")
logging.info(f"Prediction horizon: {OUTPUT_CHUNK_LENGTH}")

[INFO] Loaded hyperparameters from LSTM tuning:
[INFO]   BATCH_SIZE: 128 (from LSTM)
[INFO]   BASE_LR: 2e-05 (LSTM lr / 50)
[INFO]   CONTEXT_LENGTH: 24
[INFO]   GRAD_ACCUMULATION: 4 (effective batch: 512)
[INFO] 
[INFO] Warmup strategy:
[INFO]   Phase 1: 10 epochs, freeze all but head, LR=2e-05
[INFO]   Phase 2: 10 epochs, unfreeze 2 layers, LR=1e-05
[INFO] 
[INFO] Target columns: ['high', 'low', 'close', 'volume']
[INFO] Past covariates: 27 features
[INFO] Future covariates: 6 features
[INFO] Prediction horizon: 10


## 2. Data Loading (Like AutoGluon)

In [3]:
# Get common files between train and val (like AutoGluon)
train_files = {f for f in os.listdir(TRAIN_DIR) if f.endswith('.parquet')}
val_files = {f for f in os.listdir(VAL_DIR) if f.endswith('.parquet')}
common_files = sorted(list(train_files.intersection(val_files)))

random.seed(SEED)
random.shuffle(common_files)
files_to_use = common_files[:MAX_NUMBER_FILES]

logging.info(f"Found {len(common_files)} common files")
logging.info(f"Using {len(files_to_use)} files for training/validation")

[INFO] Found 672 common files
[INFO] Using 10 files for training/validation


## 3. Build Items for Chronos2

In [4]:
def prepare_df_for_chronos(df: pd.DataFrame):
    """Prepare dataframe for Chronos2."""
    df = df.copy()
    
    # Handle timestamp
    if 'ExecutionTime' in df.columns:
        df['ExecutionTime'] = pd.to_datetime(df['ExecutionTime'])
        if df['ExecutionTime'].dt.tz is not None:
            df['ExecutionTime'] = df['ExecutionTime'].dt.tz_localize(None)
        df = df.sort_values('ExecutionTime')
    
    return df

def _build_item_for_file(df: pd.DataFrame, for_training: bool = False, prediction_length: int = 10):
    """
    Build a multivariate Chronos2 item from a single file.

    If for_training is True, future_covariates are sliced to prediction_length as a workaround
    for a bug in the Chronos library's fit method.
    """
    targets_used = []
    tensors = []

    # Stack all 4 targets together
    for c in TARGET_COLS:
        if c not in df.columns:
            continue

        s = pd.to_numeric(df[c], errors="coerce").astype(np.float32)

        if c == "volume" and not np.isfinite(s.to_numpy()).any():
            s = s.fillna(0.0).astype(np.float32)

        arr = s.to_numpy()
        if np.isfinite(arr).any():
            targets_used.append(c)
            tensors.append(arr)

    if not targets_used:
        return None, []

    tgt = np.vstack([t.astype(np.float32, copy=False) for t in tensors])

    past_cov = {}
    for c in PAST_COVARIATES:
        if c in df.columns:
            s = df[c]
            if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
                past_cov[c] = s.to_numpy(dtype=np.float32)
            elif pd.api.types.is_object_dtype(s):
                past_cov[c] = s.to_numpy()

    future_cov = {}
    for c in FUTURE_COVARIATES:
        if c in df.columns:
            s = df[c]
            arr = s.to_numpy()
            if for_training:
                # Workaround for fit method bug
                future_cov[c] = arr[-prediction_length:]
            else:
                future_cov[c] = arr

    item = {
        "target": tgt,
        "past_covariates": past_cov,
        "future_covariates": future_cov
    }

    return item, targets_used

def load_data_for_chronos(directory, files_list, for_training=False, prediction_length=10):
    """Load parquet files and build Chronos2 items."""
    items = []
    item_ids = []

    for file_name in tqdm(files_list, desc=f"Loading from {os.path.basename(directory)}"):
        file_path = os.path.join(directory, file_name)
        df = pd.read_parquet(file_path)
        df = prepare_df_for_chronos(df)

        item, targets = _build_item_for_file(df, for_training=for_training, prediction_length=prediction_length)
        if item is not None and len(targets) > 0:
            items.append(item)
            item_ids.append(file_name.replace('.parquet', ''))

    logging.info(f"Loaded {len(items)} items from {os.path.basename(directory)}")
    if items:
        logging.info(f"  Sample target shape: {items[0]['target'].shape}")
        logging.info(f"  Past covariates: {len(items[0]['past_covariates'])} features")
        logging.info(f"  Future covariates: {len(items[0]['future_covariates'])} features")

    return items, item_ids

In [5]:
# Load training and validation data
train_items, train_ids = load_data_for_chronos(
    TRAIN_DIR, files_to_use, 
    for_training=True, 
    prediction_length=OUTPUT_CHUNK_LENGTH
)
val_items, val_ids = load_data_for_chronos(
    VAL_DIR, files_to_use,
    for_training=False
)

logging.info(f"\nTrain items: {len(train_items)}")
logging.info(f"Val items: {len(val_items)}")

Loading from train:   0%|          | 0/10 [00:00<?, ?it/s]

[INFO] Loaded 10 items from train
[INFO]   Sample target shape: (4, 69517)
[INFO]   Past covariates: 27 features
[INFO]   Future covariates: 6 features


Loading from val:   0%|          | 0/10 [00:00<?, ?it/s]

[INFO] Loaded 10 items from val
[INFO]   Sample target shape: (4, 34360)
[INFO]   Past covariates: 27 features
[INFO]   Future covariates: 6 features
[INFO] 
Train items: 10
[INFO] Val items: 10


## 4. Helper Functions for Evaluation

In [6]:
# ============================================================================
# Helper Functions
# ============================================================================

def model_parameter_sum(pipeline):
    """Compute checksum of model parameters for tracking changes."""
    total = 0.0
    for param in pipeline.model.parameters():
        total += float(param.detach().abs().sum().cpu())
    return total

def _latest_checkpoint_dir(outdir: str) -> str:
    """Find the latest checkpoint directory."""
    import glob
    ckpts = sorted(glob.glob(os.path.join(outdir, "checkpoint-*")))
    return ckpts[-1] if ckpts else outdir
    
def slice_future_covariates(items, prediction_length):
    """
    Creates a deep copy of the items and slices the future_covariates to the prediction_length.
    This is a workaround for a bug in the Chronos library's fit method.
    """
    import copy

    new_items = copy.deepcopy(items)
    for item in new_items:
        if "future_covariates" in item:
            for key, value in item["future_covariates"].items():
                item["future_covariates"][key] = value[-prediction_length:]
    return new_items

def train_burst_with_regularization(pipe, train_items, val_items, steps, lr, batch_size,
                                    ctx_len, pred_len, outdir, seed=SEED,
                                    weight_decay=1e-4, grad_accum=4, clip_val=1.0):
    """
    Train one burst with weight decay, gradient clipping, and gradient accumulation.
    Uses HuggingFace-style kwargs with graceful fallbacks (copied from colleague's approach).

    Returns: New pipeline reloaded from checkpoint
    """
    checksum_before = model_parameter_sum(pipe)
    pipe.model.train()
    torch.set_grad_enabled(True)

    # WORKAROUND for bug in Chronos library
    train_items_sliced = slice_future_covariates(train_items, pred_len)
    val_items_sliced = slice_future_covariates(val_items, pred_len)

    # Base arguments that should always work
    base_kwargs = dict(
        inputs=train_items_sliced,
        prediction_length=pred_len,
        validation_inputs=(val_items_sliced if len(val_items_sliced) > 0 else None),
        learning_rate=lr,
        batch_size=batch_size,
        context_length=ctx_len,
        output_dir=outdir,
        num_steps=int(steps),
        seed=seed,
        weight_decay=weight_decay,
    )

    # Try full feature set with gradient clipping and accumulation
    extras = {}
    extras["gradient_accumulation_steps"] = int(grad_accum)
    if clip_val is not None:
        extras["max_grad_norm"] = float(clip_val)

    # Graceful degradation: try with all features, then drop unsupported ones
    try:
        logging.info(f"  Attempting training with weight_decay={weight_decay}, "
                     f"grad_accum={grad_accum}, max_grad_norm={clip_val}")
        pipe.fit(**base_kwargs, **extras)
    except TypeError as e:
        logging.warning(f"  max_grad_norm not supported, retrying without it: {e}")
        extras.pop("max_grad_norm", None)
        try:
            pipe.fit(**base_kwargs, **extras)
        except TypeError as e:
            logging.warning(f"  gradient_accumulation_steps not supported, using basic fit: {e}")
            extras.pop("gradient_accumulation_steps", None)
            pipe.fit(**base_kwargs)

    # Reload checkpoint to get updated weights
    ckpt_dir = _latest_checkpoint_dir(outdir)
    new_pipe = BaseChronosPipeline.from_pretrained(
        ckpt_dir,
        device_map="cuda" if torch.cuda.is_available() else None
    )

    checksum_after = model_parameter_sum(new_pipe)
    logging.info(f"  Model checksum delta: {checksum_after - checksum_before:.8f}")

    return new_pipe

def freeze_all_layers(pipeline):
    """Freeze all model parameters."""
    for param in pipeline.model.parameters():
        param.requires_grad = False
    logging.info("Froze all layers")

def unfreeze_head_only(pipeline):
    """
    Unfreeze only the output quantile head (output_patch_embedding).
    
    Chronos-2 Architecture:
    - input_patch_embedding: Input tokenization
    - encoder: Transformer blocks
    - output_patch_embedding: Quantile head + upscaling (THIS IS THE HEAD!)
    - shared: Shared embeddings
    """
    freeze_all_layers(pipeline)
    unfrozen_count = 0
    
    # Unfreeze output_patch_embedding (the actual Chronos-2 quantile head)
    for name, param in pipeline.model.named_parameters():
        if name.startswith('output_patch_embedding'):
            param.requires_grad = True
            unfrozen_count += param.numel()
    
    logging.info(f"Unfroze quantile head (output_patch_embedding): ~{unfrozen_count:,} parameters")
    return unfrozen_count > 0

def unfreeze_n_encoder_blocks(pipeline, n_blocks=2):
    """
    Unfreeze the last n encoder blocks (in addition to currently unfrozen params).
    
    Chronos-2 has 12 encoder blocks (0-11). This will unfreeze the last n blocks.
    """
    unfrozen_count = 0
    try:
        # Find max block number
        max_block = -1
        for name, _ in pipeline.model.named_parameters():
            if 'encoder.block.' in name:
                parts = name.split('.')
                for i, part in enumerate(parts):
                    if part == 'block' and i + 1 < len(parts):
                        try:
                            block_num = int(parts[i + 1])
                            max_block = max(max_block, block_num)
                        except ValueError:
                            pass
        
        if max_block >= 0:
            # Unfreeze last n_blocks
            blocks_to_unfreeze = list(range(max(0, max_block - n_blocks + 1), max_block + 1))
            logging.info(f"Unfreezing encoder blocks: {blocks_to_unfreeze}")
            
            for name, param in pipeline.model.named_parameters():
                for block_num in blocks_to_unfreeze:
                    if f'encoder.block.{block_num}.' in name:
                        if not param.requires_grad:
                            param.requires_grad = True
                            unfrozen_count += param.numel()
                        break
            
            logging.info(f"Unfroze last {n_blocks} encoder blocks: ~{unfrozen_count:,} parameters")
        else:
            logging.warning("Could not find encoder blocks, unfreezing all")
            for param in pipeline.model.parameters():
                if not param.requires_grad:
                    param.requires_grad = True
                    unfrozen_count += param.numel()
    except Exception as e:
        logging.warning(f"Layer unfreezing failed: {e}, unfreezing all")
        for param in pipeline.model.parameters():
            if not param.requires_grad:
                param.requires_grad = True
                unfrozen_count += param.numel()
    
    return unfrozen_count > 0

def show_trainable_params(pipeline):
    """Display trainable parameter count."""
    total = 0
    trainable = 0
    for param in pipeline.model.parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    logging.info(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
    return trainable, total

def _normalize_CH(arr, n_channels_hint=None):
    """Normalize array to (C, H) shape."""
    arr = np.asarray(arr)
    if arr.ndim == 1:
        return arr.reshape(1, -1)
    
    C, H = arr.shape[0], arr.shape[1]
    if n_channels_hint is None:
        return arr if C <= H else arr.T
    
    if C == n_channels_hint:
        return arr
    if H == n_channels_hint:
        return arr.T
    
    return arr if C <= H else arr.T

def evaluate_forecasts(pipeline, val_items, context_length, pred_len, target_cols):
    """
    Evaluates forecasts using a sliding window approach on the validation data.
    Batches predictions for efficiency.
    """
    all_preds = []
    all_actuals = []

    # Create batches of windows to predict on
    predict_inputs = []
    for item in val_items:
        target = item['target']
        past_covariates = item.get('past_covariates')
        future_covariates = item.get('future_covariates')

        for i in range(0, target.shape[1] - context_length - pred_len + 1, pred_len):
            context = target[:, i : i + context_length]
            actual = target[:, i + context_length : i + context_length + pred_len]
            
            past_covariates_context = None
            if past_covariates is not None:
                past_covariates_context = {k: v[i : i + context_length] for k, v in past_covariates.items()}
            
            future_covariates_context = None
            if future_covariates is not None:
                future_covariates_context = {k: v[i + context_length : i + context_length + pred_len] for k, v in future_covariates.items()}

            predict_input = {"target": context}
            if past_covariates_context:
                predict_input["past_covariates"] = past_covariates_context
            if future_covariates_context:
                predict_input["future_covariates"] = future_covariates_context

            predict_inputs.append(predict_input)
            all_actuals.append(actual)

    # Batch predict
    if not predict_inputs:
        return None

    logging.info(f"Predicting {len(predict_inputs)} windows in batches...")

    forecasts = pipeline.predict(
        predict_inputs,
        prediction_length=pred_len,
        batch_size=BATCH_SIZE
    )
    
    all_preds = forecasts # forecasts is already a list of numpy arrays

    # Calculate sMAPE
    all_preds_tensor = torch.from_numpy(np.array(all_preds))
    all_actuals_tensor = torch.from_numpy(np.array(all_actuals))

    # extract target forecasts
    all_preds_tensor = all_preds_tensor[:, :, 0, :]

    # Permute to (num_windows, pred_length, num_targets) for the metric
    all_preds_tensor = all_preds_tensor.permute(0, 2, 1)
    all_actuals_tensor = all_actuals_tensor.permute(0, 2,1)

    smape = smoothed_smape(all_actuals_tensor, all_preds_tensor)

    # Per-target sMAPE
    per_target_smape = {}
    for i, target_name in enumerate(target_cols):
        per_target_smape[target_name] = smoothed_smape(all_actuals_tensor[:, :, i], all_preds_tensor[:, :, i])


    return {
        'overall_smape_%': smape.item() * 100,
        'per_target': {k: v.item() * 100 for k, v in per_target_smape.items()},
        'n_predictions': len(all_preds),
    }

## 5. Zero-Shot Evaluation

In [7]:
# Load base Chronos2 pipeline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda" if torch.cuda.is_available() else None
)
logging.info(f"Loaded Chronos2 pipeline on device: {device}")

[INFO] Loaded Chronos2 pipeline on device: cuda


In [8]:
# ============================================================================
# Zero-Shot Baseline Evaluation
# ============================================================================
logging.info("=" * 80)
logging.info("ZERO-SHOT BASELINE EVALUATION")
logging.info("=" * 80)

# Evaluate using proper train/val separation
results_zero_shot = evaluate_forecasts(pipeline, val_items, INPUT_CHUNK_LENGTH, OUTPUT_CHUNK_LENGTH, TARGET_COLS)

if results_zero_shot:
    logging.info(f"\nZero-Shot sMAPE: {results_zero_shot['overall_smape_%']:.2f}%")
    logging.info(f"Predictions made: {results_zero_shot['n_predictions']}")
    print("\nPer-target sMAPE:")
    print(results_zero_shot['per_target'])
    
    # Store baseline for comparison
    baseline_smape = results_zero_shot['overall_smape_%']
    logging.info(f"\n{'='*80}")
    logging.info(f"BASELINE TO BEAT: {baseline_smape:.2f}%")
    logging.info(f"{'='*80}\n")
else:
    baseline_smape = float('inf')
    logging.error("Zero-shot evaluation failed!")

[INFO] ================================================================================
[INFO] ZERO-SHOT BASELINE EVALUATION
[INFO] ================================================================================
[INFO] Predicting 34330 windows in batches...
[INFO] 
Zero-Shot sMAPE: 147.69%
[INFO] Predictions made: 34330
[INFO] 
[INFO] BASELINE TO BEAT: 147.69%
[INFO] ================================================================================




Per-target sMAPE:
{'high': 144.80396509170532, 'low': 144.9514865875244, 'close': 144.86151933670044, 'volume': 156.13645315170288}


## 6. Fine-Tuning

In [9]:
# ============================================================================
# Warmup Phase 1: Train Head Only
# ============================================================================
logging.info("=" * 80)
logging.info("WARMUP PHASE 1: Training head only for stability")
logging.info("=" * 80)

# Reload pipeline for fine-tuning
pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cuda" if torch.cuda.is_available() else None
)
pipeline.model.train()
torch.set_grad_enabled(True)

# Freeze all layers except head
unfreeze_head_only(pipeline)
show_trainable_params(pipeline)

# Setup directories
WARMUP1_DIR = os.path.join(MODELS_DIR, "chronos2_warmup_phase1")
if os.path.exists(WARMUP1_DIR):
    shutil.rmtree(WARMUP1_DIR)
os.makedirs(WARMUP1_DIR, exist_ok=True)

# Calculate steps from epochs
estimated_batches_per_epoch = max(1, len(train_items) // BATCH_SIZE)
warmup1_steps = WARMUP_EPOCHS_PHASE1 * estimated_batches_per_epoch

logging.info(f"Training configuration:")
logging.info(f"  Training items: {len(train_items)}")
logging.info(f"  Validation items: {len(val_items)}")
logging.info(f"  Epochs: {WARMUP_EPOCHS_PHASE1}")
logging.info(f"  Estimated steps: {warmup1_steps}")
logging.info(f"  Batch size: {BATCH_SIZE}")
logging.info(f"  Learning rate: {WARMUP_LR_PHASE1}")
logging.info(f"  Weight decay: {WEIGHT_DECAY}")
logging.info(f"  Gradient clip: {GRADIENT_CLIP}")
logging.info(f"  Gradient accumulation: {GRAD_ACCUMULATION} (effective batch: {BATCH_SIZE * GRAD_ACCUMULATION})")

# Train with frozen encoder using regularization
logging.info("\nStarting Warmup Phase 1 training with weight decay, gradient clipping, and gradient accumulation...")
pipeline = train_burst_with_regularization(
    pipe=pipeline,
    train_items=train_items,
    val_items=val_items,
    steps=warmup1_steps,
    lr=WARMUP_LR_PHASE1,
    batch_size=BATCH_SIZE,
    ctx_len=INPUT_CHUNK_LENGTH,
    pred_len=OUTPUT_CHUNK_LENGTH,
    outdir=WARMUP1_DIR,
    seed=SEED,
    weight_decay=WEIGHT_DECAY,
    grad_accum=GRAD_ACCUMULATION,
    clip_val=GRADIENT_CLIP
)
logging.info("Warmup Phase 1 complete!")

[INFO] ================================================================================
[INFO] WARMUP PHASE 1: Training head only for stability
[INFO] ================================================================================
[INFO] Froze all layers
[INFO] Unfroze quantile head (output_patch_embedding): ~3,653,280 parameters
[INFO] Trainable parameters: 3,653,280 / 119,477,664 (3.1%)
[INFO] Training configuration:
[INFO]   Training items: 10
[INFO]   Validation items: 10
[INFO]   Epochs: 10
[INFO]   Estimated steps: 10
[INFO]   Batch size: 128
[INFO]   Learning rate: 2e-05
[INFO]   Weight decay: 0.0001
[INFO]   Gradient clip: 1.0
[INFO]   Gradient accumulation: 4 (effective batch: 512)
[INFO] 
Starting Warmup Phase 1 training with weight decay, gradient clipping, and gradient accumulation...
[INFO]   Attempting training with weight_decay=0.0001, grad_accum=4, max_grad_norm=1.0
C:\Users\merta\AppData\Local\Temp\ipykernel_25304\1763153478.py:73: FutureWarning: Fine-tuning support i

Step,Training Loss,Validation Loss
10,0.341300,0.101973


[INFO] Finetuned model saved to C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_warmup_phase1\finetuned-ckpt
[INFO]   Model checksum delta: 11.15440559
[INFO] Warmup Phase 1 complete!


In [10]:
# ============================================================================
# Warmup Phase 2: Unfreeze 2 Encoder Blocks
# ============================================================================
logging.info("\n" + "=" * 80)
logging.info("WARMUP PHASE 2: Unfreezing 2 additional encoder blocks")
logging.info("=" * 80)

# Unfreeze head (was frozen on reload) + 2 encoder blocks
unfreeze_head_only(pipeline)
unfreeze_n_encoder_blocks(pipeline, n_blocks=2)
show_trainable_params(pipeline)

# Setup directory
WARMUP2_DIR = os.path.join(MODELS_DIR, "chronos2_warmup_phase2")
if os.path.exists(WARMUP2_DIR):
    shutil.rmtree(WARMUP2_DIR)
os.makedirs(WARMUP2_DIR, exist_ok=True)

# Calculate steps
warmup2_steps = WARMUP_EPOCHS_PHASE2 * estimated_batches_per_epoch

logging.info(f"Training configuration:")
logging.info(f"  Epochs: {WARMUP_EPOCHS_PHASE2}")
logging.info(f"  Estimated steps: {warmup2_steps}")
logging.info(f"  Batch size: {BATCH_SIZE}")
logging.info(f"  Learning rate: {WARMUP_LR_PHASE2} (reduced from phase 1)")

# Train using regularization
logging.info("\nStarting Warmup Phase 2 training with weight decay, gradient clipping, and gradient accumulation...")
pipeline = train_burst_with_regularization(
    pipe=pipeline,
    train_items=train_items,
    val_items=val_items,
    steps=warmup2_steps,
    lr=WARMUP_LR_PHASE2,
    batch_size=BATCH_SIZE,
    ctx_len=INPUT_CHUNK_LENGTH,
    pred_len=OUTPUT_CHUNK_LENGTH,
    outdir=WARMUP2_DIR,
    seed=SEED,
    weight_decay=WEIGHT_DECAY,
    grad_accum=GRAD_ACCUMULATION,
    clip_val=GRADIENT_CLIP
)
logging.info("Warmup Phase 2 complete!")

[INFO] 
[INFO] WARMUP PHASE 2: Unfreezing 2 additional encoder blocks
[INFO] ================================================================================
[INFO] Froze all layers
[INFO] Unfroze quantile head (output_patch_embedding): ~3,653,280 parameters
[INFO] Unfreezing encoder blocks: [10, 11]
[INFO] Unfroze last 2 encoder blocks: ~18,878,976 parameters
[INFO] Trainable parameters: 22,532,256 / 119,477,664 (18.9%)
[INFO] Training configuration:
[INFO]   Epochs: 10
[INFO]   Estimated steps: 10
[INFO]   Batch size: 128
[INFO]   Learning rate: 1e-05 (reduced from phase 1)
[INFO] 
Starting Warmup Phase 2 training with weight decay, gradient clipping, and gradient accumulation...
[INFO]   Attempting training with weight_decay=0.0001, grad_accum=4, max_grad_norm=1.0
C:\Users\merta\AppData\Local\Temp\ipykernel_25304\1763153478.py:73: FutureWarning: Fine-tuning support is experimental and may be changed in future versions.
  pipe.fit(**base_kwargs, **extras)
Could not estimate the numbe

Step,Training Loss,Validation Loss
10,0.327200,0.118667


[INFO] Finetuned model saved to C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_warmup_phase2\finetuned-ckpt
[INFO]   Model checksum delta: 1.81957769
[INFO] Warmup Phase 2 complete!


In [11]:
# ============================================================================
# Post-Warmup Evaluation
# ============================================================================
logging.info("\n" + "=" * 80)
logging.info("POST-WARMUP EVALUATION")
logging.info("=" * 80)

# Pipeline is already reloaded by train_burst_with_regularization
logging.info("Evaluating post-warmup model...")

# Evaluate
results_warmup = evaluate_forecasts(pipeline, val_items, INPUT_CHUNK_LENGTH, OUTPUT_CHUNK_LENGTH, TARGET_COLS)

if results_warmup:
    logging.info(f"\nPost-Warmup sMAPE: {results_warmup['overall_smape_%']:.2f}%")
    logging.info(f"Predictions made: {results_warmup['n_predictions']}")
    print("\nPer-target sMAPE:")
    print(results_warmup['per_target'])
    
    warmup_smape = results_warmup['overall_smape_%']
    improvement_from_baseline = baseline_smape - warmup_smape
    
    logging.info(f"\n{'='*80}")
    logging.info(f"WARMUP RESULTS:")
    logging.info(f"  Baseline:    {baseline_smape:.2f}%")
    logging.info(f"  After Warmup: {warmup_smape:.2f}%")
    logging.info(f"  Improvement:  {improvement_from_baseline:+.2f}%")
    logging.info(f"{'='*80}\n")
else:
    warmup_smape = float('inf')
    logging.error("Post-warmup evaluation failed!")
    
# Store best model checkpoint and sMAPE so far
best_checkpoint_path = _latest_checkpoint_dir(WARMUP2_DIR)
lowest_smape_score = warmup_smape

[INFO] 
[INFO] POST-WARMUP EVALUATION
[INFO] ================================================================================
[INFO] Evaluating post-warmup model...
[INFO] Predicting 34330 windows in batches...
[INFO] 
Post-Warmup sMAPE: 34.87%
[INFO] Predictions made: 34330
[INFO] 
[INFO] WARMUP RESULTS:
[INFO]   Baseline:    147.69%
[INFO]   After Warmup: 34.87%
[INFO]   Improvement:  +112.82%
[INFO] ================================================================================




Per-target sMAPE:
{'high': 31.740012764930725, 'low': 31.75986409187317, 'close': 31.718912720680237, 'volume': 44.26029920578003}


## 7. Burst Training with Early Stopping

In [12]:
# ============================================================================
# Burst Training Loop with LR Decay and Early Stopping
# ============================================================================
logging.info("=" * 80)
logging.info("BURST TRAINING")
logging.info("=" * 80)

# Training state
current_learning_rate = BASE_LR
bursts_without_improvement = 0
burst_history = []

# Unfreeze all layers for burst training
pipeline.model.train()
torch.set_grad_enabled(True)
for param in pipeline.model.parameters():
    param.requires_grad = True
show_trainable_params(pipeline)

logging.info(f"Burst training configuration:")
logging.info(f"  Steps per burst: {BURST_STEPS}")
logging.info(f"  Max bursts: {MAX_BURSTS}")
logging.info(f"  Initial LR: {current_learning_rate}")
logging.info(f"  LR decay factor: {LR_DECAY_FACTOR}")
logging.info(f"  Min LR: {MIN_LR}")
logging.info(f"  Patience: {PATIENCE} bursts")
logging.info(f"  Weight decay: {WEIGHT_DECAY}")
logging.info(f"  Gradient clip: {GRADIENT_CLIP}")
logging.info(f"  Gradient accumulation: {GRAD_ACCUMULATION}")
logging.info(f"  Early stop if no improvement for {PATIENCE} consecutive bursts")

for burst_idx in range(MAX_BURSTS):
    logging.info(f"\n{'='*60}")
    logging.info(f"BURST {burst_idx + 1}/{MAX_BURSTS}")
    logging.info(f"  Learning rate: {current_learning_rate}")
    logging.info(f"  Bursts without improvement: {bursts_without_improvement}/{PATIENCE}")
    logging.info(f"{'='*60}")
    
    # Setup burst directory
    burst_dir = os.path.join(MODELS_DIR, f"chronos2_burst_{burst_idx + 1}")
    if os.path.exists(burst_dir):
        shutil.rmtree(burst_dir)
    os.makedirs(burst_dir, exist_ok=True)
    
    try:
        # Train for one burst with regularization
        logging.info(f"Training burst {burst_idx + 1} with weight decay, gradient clipping, and accumulation...")
        pipeline = train_burst_with_regularization(
            pipe=pipeline,
            train_items=train_items,
            val_items=val_items,
            steps=BURST_STEPS,
            lr=current_learning_rate,
            batch_size=BATCH_SIZE,
            ctx_len=INPUT_CHUNK_LENGTH,
            pred_len=OUTPUT_CHUNK_LENGTH,
            outdir=burst_dir,
            seed=SEED + burst_idx,
            weight_decay=WEIGHT_DECAY,
            grad_accum=GRAD_ACCUMULATION,
            clip_val=GRADIENT_CLIP
        )
        
        # Evaluate
        logging.info(f"Evaluating burst {burst_idx + 1}...")
        burst_results = evaluate_forecasts(pipeline, val_items, INPUT_CHUNK_LENGTH, OUTPUT_CHUNK_LENGTH, TARGET_COLS)
        
        if burst_results:
            burst_smape = burst_results['overall_smape_%']
            logging.info(f"Burst {burst_idx + 1} sMAPE: {burst_smape:.2f}%")
            
            # Check for improvement
            improvement = lowest_smape_score - burst_smape
            
            if improvement > 0.5:  # Improvement threshold of 0.5%
                logging.info(f"✓ Improvement: {improvement:.2f}% (New best: {burst_smape:.2f}%)")
                lowest_smape_score = burst_smape
                best_checkpoint_path = _latest_checkpoint_dir(burst_dir)
                bursts_without_improvement = 0
            else:
                logging.info(f"✗ No improvement (Best: {lowest_smape_score:.2f}%)")
                bursts_without_improvement += 1
                
                # Reduce learning rate on plateau
                if bursts_without_improvement > 0:
                    old_lr = current_learning_rate
                    current_learning_rate = max(MIN_LR, current_learning_rate * LR_DECAY_FACTOR)
                    logging.info(f"Reducing LR: {old_lr:.6f} → {current_learning_rate:.6f}")
            
            # Store history
            burst_history.append({
                'burst': burst_idx + 1,
                'smape': burst_smape,
                'lr': current_learning_rate,
                'is_best': improvement > 0.5
            })
            
            # Early stopping check
            if bursts_without_improvement >= PATIENCE:
                logging.info(f"\n{'='*60}")
                logging.info(f"EARLY STOPPING: No improvement for {PATIENCE} bursts")
                logging.info(f"{'='*60}")
                break
        else:
            logging.error(f"Burst {burst_idx + 1} evaluation failed!")
            bursts_without_improvement += 1
    
    except Exception as e:
        logging.error(f"Burst {burst_idx + 1} training failed: {e}")
        bursts_without_improvement += 1
        
        if bursts_without_improvement >= PATIENCE:
            logging.error("Stopping due to repeated failures")
            break

# Summary
logging.info(f"\n{'='*80}")
logging.info("BURST TRAINING SUMMARY")
logging.info(f"{'='*80}")
logging.info(f"Total bursts completed: {len(burst_history)}")
logging.info(f"Best sMAPE achieved: {lowest_smape_score:.2f}%")
logging.info(f"Best checkpoint: {best_checkpoint_path}")

if burst_history:
    print("\nBurst history:")
    burst_df = pd.DataFrame(burst_history)
    print(burst_df.to_string(index=False))

[INFO] ================================================================================
[INFO] BURST TRAINING
[INFO] ================================================================================
[INFO] Trainable parameters: 119,477,664 / 119,477,664 (100.0%)
[INFO] Burst training configuration:
[INFO]   Steps per burst: 200
[INFO]   Max bursts: 20
[INFO]   Initial LR: 2e-05
[INFO]   LR decay factor: 0.75
[INFO]   Min LR: 2.0000000000000002e-07
[INFO]   Patience: 5 bursts
[INFO]   Weight decay: 0.0001
[INFO]   Gradient clip: 1.0
[INFO]   Gradient accumulation: 4
[INFO]   Early stop if no improvement for 5 consecutive bursts
[INFO] 
[INFO] BURST 1/20
[INFO]   Learning rate: 2e-05
[INFO]   Bursts without improvement: 0/5
[INFO] ============================================================
[INFO] Training burst 1 with weight decay, gradient clipping, and accumulation...
[INFO]   Attempting training with weight_decay=0.0001, grad_accum=4, max_grad_norm=1.0
C:\Users\merta\AppData\Local\Tem

Step,Training Loss,Validation Loss
100,0.205900,0.100344
200,0.134800,0.104367


[INFO] Finetuned model saved to C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_burst_1\finetuned-ckpt
[INFO]   Model checksum delta: 216.33272028
[INFO] Evaluating burst 1...
[INFO] Predicting 34330 windows in batches...
[INFO] Burst 1 sMAPE: 36.82%
[INFO] ✗ No improvement (Best: 34.87%)
[INFO] Reducing LR: 0.000020 → 0.000015
[INFO] 
[INFO] BURST 2/20
[INFO]   Learning rate: 1.5000000000000002e-05
[INFO]   Bursts without improvement: 1/5
[INFO] ============================================================
[INFO] Training burst 2 with weight decay, gradient clipping, and accumulation...
[INFO]   Attempting training with weight_decay=0.0001, grad_accum=4, max_grad_norm=1.0
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss
100,0.187200,0.095246
200,0.147500,0.091155


[INFO] Finetuned model saved to C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_burst_2\finetuned-ckpt
[INFO]   Model checksum delta: 88.29306602
[INFO] Evaluating burst 2...
[INFO] Predicting 34330 windows in batches...
[INFO] Burst 2 sMAPE: 36.66%
[INFO] ✗ No improvement (Best: 34.87%)
[INFO] Reducing LR: 0.000015 → 0.000011
[INFO] 
[INFO] BURST 3/20
[INFO]   Learning rate: 1.1250000000000002e-05
[INFO]   Bursts without improvement: 2/5
[INFO] ============================================================
[INFO] Training burst 3 with weight decay, gradient clipping, and accumulation...
[INFO]   Attempting training with weight_decay=0.0001, grad_accum=4, max_grad_norm=1.0
Could not estimate the number of tokens of the input, floating-point operations will not be computed


...

Step,Training Loss,Validation Loss
100,0.150800,0.108758
200,0.164700,0.107624


[INFO] Finetuned model saved to C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_burst_4\finetuned-ckpt
[INFO]   Model checksum delta: 31.88218927
[INFO] Evaluating burst 4...
[INFO] Predicting 34330 windows in batches...
[INFO] Burst 4 sMAPE: 36.91%
[INFO] ✗ No improvement (Best: 34.87%)
[INFO] Reducing LR: 0.000008 → 0.000006
[INFO] 
[INFO] BURST 5/20
[INFO]   Learning rate: 6.328125000000001e-06
[INFO]   Bursts without improvement: 4/5
[INFO] ============================================================
[INFO] Training burst 5 with weight decay, gradient clipping, and accumulation...
[INFO]   Attempting training with weight_decay=0.0001, grad_accum=4, max_grad_norm=1.0
Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss,Validation Loss
100,0.145600,0.105481
200,0.117600,0.109832


[INFO] Finetuned model saved to C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_burst_5\finetuned-ckpt
[INFO]   Model checksum delta: 18.92890024
[INFO] Evaluating burst 5...
[INFO] Predicting 34330 windows in batches...
[INFO] Burst 5 sMAPE: 37.77%
[INFO] ✗ No improvement (Best: 34.87%)
[INFO] Reducing LR: 0.000006 → 0.000005
[INFO] 
[INFO] EARLY STOPPING: No improvement for 5 bursts
[INFO] ============================================================
[INFO] 
[INFO] BURST TRAINING SUMMARY
[INFO] ================================================================================
[INFO] Total bursts completed: 5
[INFO] Best sMAPE achieved: 34.87%
[INFO] Best checkpoint: C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_warmup_phase2\checkpoint-10



Burst history:
 burst     smape       lr  is_best
     1 36.819452 0.000015    False
     2 36.659667 0.000011    False
     3 37.661120 0.000008    False
     4 36.909130 0.000006    False
     5 37.765941 0.000005    False


## 8. Final Evaluation and Comparison

In [13]:
# ============================================================================
# Load Best Model and Final Evaluation
# ============================================================================

# Reload best model
pipeline_best = BaseChronosPipeline.from_pretrained(
    best_checkpoint_path,
    device_map="cuda" if torch.cuda.is_available() else None
)
logging.info(f"Loaded best model from: {best_checkpoint_path}")

final_smape = lowest_smape_score

# ============================================================================
# Evaluation on a Sample of the Test Set
# ============================================================================
logging.info("\n" + "=" * 80)
logging.info("EVALUATION ON TEST SET SAMPLE")
logging.info("=" * 80)

# Get 5 random test files
all_test_files = [f for f in os.listdir(TEST_DIR) if f.endswith('.parquet')]
random.seed(SEED)
if len(all_test_files) > 5:
    test_files_sample = random.sample(all_test_files, 5)
    logging.info(f"Randomly selected 5 files from the test set for final evaluation.")
else:
    test_files_sample = all_test_files
    logging.info(f"Using all {len(all_test_files)} files from the test set as it's less than or equal to 5.")

# Load test data
test_items, test_ids = load_data_for_chronos(
    TEST_DIR,
    test_files_sample,
    for_training=False
)

# Evaluate on the test set sample
results_test = None
if test_items:
    results_test = evaluate_forecasts(pipeline_best, test_items,
        INPUT_CHUNK_LENGTH, OUTPUT_CHUNK_LENGTH, TARGET_COLS)
else:
    logging.warning("No test items were loaded, skipping test set evaluation.")

if results_test:
    test_smape = results_test['overall_smape_%']
    logging.info(f"\nTest Set Sample sMAPE: {test_smape:.2f}%")
    logging.info(f"Predictions made: {results_test['n_predictions']}")
    print("\nPer-target sMAPE on Test Set:")
    print(results_test['per_target'])
else:
    test_smape = float('inf')
    logging.error("Test set evaluation failed or was skipped!")


# ============================================================================
# Comparison and Results Summary
# ============================================================================
logging.info("\n" + "=" * 80)
logging.info("OVERALL COMPARISON")
logging.info("=" * 80)

print(f"\n{'Stage':<25} {'sMAPE':<12} {'Improvement':<15}")
print("-" * 52)
print(f"{'Zero-Shot (Baseline)':<25} {baseline_smape:>10.2f}%  {'-':<15}")

warmup_improvement = baseline_smape - warmup_smape
print(f"{'After Warmup':<25} {warmup_smape:>10.2f}%  {warmup_improvement:>+10.2f}%")

final_improvement = baseline_smape - final_smape
print(f"{'After Burst Training (Val)':<25} {final_smape:>10.2f}%  {final_improvement:>+10.2f}%")

test_improvement = baseline_smape - test_smape
print(f"{'Final Model on Test Set':<25} {test_smape:>10.2f}%  {test_improvement:>+10.2f}%")

# Determine best approach
if final_smape < baseline_smape:
    best_approach = 'fine_tuned'
    improvement = baseline_smape - final_smape
    logging.info(f"\n✓ Fine-tuning improved validation sMAPE by {improvement:.2f}%")
else:
    best_approach = 'zero_shot'
    logging.info(f"\n✗ Fine-tuning did not improve over baseline on validation set")

# ============================================================================
# Save Best Model
# ============================================================================
BEST_MODEL_DIR = os.path.join(MODELS_DIR, "chronos2_best_multivariate")
if os.path.exists(BEST_MODEL_DIR):
    shutil.rmtree(BEST_MODEL_DIR)

if best_approach == 'fine_tuned':
    shutil.copytree(best_checkpoint_path, BEST_MODEL_DIR)
    logging.info(f"\nSaved fine-tuned model to: {BEST_MODEL_DIR}")
else:
    # Save zero-shot (base) model
    base_pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2")
    base_pipeline.save_pretrained(BEST_MODEL_DIR)
    logging.info(f"\nSaved zero-shot model to: {BEST_MODEL_DIR}")

# ============================================================================
# Save Results JSON
# ============================================================================
results_summary = {
    'configuration': {
        'target_columns': TARGET_COLS,
        'context_length': INPUT_CHUNK_LENGTH,
        'prediction_horizon': OUTPUT_CHUNK_LENGTH,
        'batch_size': BATCH_SIZE,
        'base_lr': BASE_LR,
        'warmup_phase1_epochs': WARMUP_EPOCHS_PHASE1,
        'warmup_phase1_lr': WARMUP_LR_PHASE1,
        'warmup_phase2_epochs': WARMUP_EPOCHS_PHASE2,
        'warmup_phase2_lr': WARMUP_LR_PHASE2,
        'burst_steps': BURST_STEPS,
        'max_bursts': MAX_BURSTS,
        'patience': PATIENCE,
        'lr_decay_factor': LR_DECAY_FACTOR,
        'weight_decay': WEIGHT_DECAY,
        'gradient_clip': GRADIENT_CLIP,
        'grad_accumulation': GRAD_ACCUMULATION,
        'n_train_items': len(train_items),
        'n_val_items': len(val_items),
    },
    'zero_shot': {
        'smape': float(baseline_smape),
    },
    'post_warmup': {
        'smape': float(warmup_smape),
    },
    'final_validation': {
        'smape': float(final_smape),
    },
    'final_test_sample': {
        'smape': float(test_smape) if results_test else None,
        'per_target': results_test['per_target'] if results_test else None,
        'n_predictions': results_test['n_predictions'] if results_test else None,
        'test_files_sampled': test_files_sample
    },
    'burst_history': burst_history,
    'best_approach': best_approach,
    'improvement_vs_baseline': float(baseline_smape - final_smape)
}

results_path = os.path.join(RESULTS_DIR, 'chronos2_multivariate_finetuning_results.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

logging.info(f"\nResults saved to: {results_path}")
logging.info("\n" + "=" * 80)
logging.info("TRAINING COMPLETE!")
logging.info("=" * 80)

[INFO] Loaded best model from: C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_warmup_phase2\checkpoint-10
[INFO] 
[INFO] EVALUATION ON TEST SET SAMPLE
[INFO] ================================================================================
[INFO] Randomly selected 5 files from the test set for final evaluation.


Loading from test:   0%|          | 0/5 [00:00<?, ?it/s]

[INFO] Loaded 5 items from test
[INFO]   Sample target shape: (4, 21798)
[INFO]   Past covariates: 27 features
[INFO]   Future covariates: 6 features
[INFO] Predicting 10885 windows in batches...
[INFO] 
Test Set Sample sMAPE: 35.61%
[INFO] Predictions made: 10885
[INFO] 
[INFO] OVERALL COMPARISON
[INFO] ================================================================================
[INFO] 
✓ Fine-tuning improved validation sMAPE by 112.82%



Per-target sMAPE on Test Set:
{'high': 31.91303312778473, 'low': 32.02900588512421, 'close': 31.93524181842804, 'volume': 46.57260477542877}

Stage                     sMAPE        Improvement    
----------------------------------------------------
Zero-Shot (Baseline)          147.69%  -              
After Warmup                   34.87%     +112.82%
After Burst Training (Val)      34.87%     +112.82%
Final Model on Test Set        35.61%     +112.08%


[INFO] 
Saved fine-tuned model to: C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\models\chronos2_best_multivariate
[INFO] 
Results saved to: C:\Users\merta\OneDrive\Desktop\FS Courses\Deep Learning\Final_Project\results\chronos2_multivariate_finetuning_results.json
[INFO] 
[INFO] TRAINING COMPLETE!
[INFO] ================================================================================
